## Creating Your Own Modules

### `torch.nn.module`

In [ ]:
# Import numpy for numerical operations
import numpy as np
# Import print_function for compatibility between Python 2 and 3
from __future__ import print_function

In [ ]:
# Import torch and neural network modules
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
# Import math for initialization and numpy for data
import math
import numpy as np

``nn.Module`` is base class for all neural network modules.

You should also write your modules as sub-class of ``nn.Module``, so that it can inherit the following attributes:

* *Recursive structure*: you can wrap an instantiation of a Module class with another one, which stores the inner one as its parent

* *Cudafiability*: you can easily cudafy the whole sequence of modules using `model.cuda()`

* *Serializable*: you can save your trained model (checkpoint, early stopping ...) using ``torch.save``, ``torch.load``

* *Parameters*: you can call model.parameters() to access all parameters at the same time. 

etc. 


#### Custom Linear module mimicking nn.Linear

In [ ]:
# Custom Linear module mimicking nn.Linear
class Linear(nn.Module):
    r"""Applies a linear transformation to the incoming data: y = Ax + b
    Args:
        in_features: size of each input sample
        out_features: size of each output sample
        bias: If set to False, the layer will not learn an additive bias. Default: True
    Shape:
        - Input: (N, in_features)
        - Output: (N, out_features)
    Attributes:
        weight: the learnable weights of the module of shape (out_features x in_features)
        bias:   the learnable bias of the module of shape (out_features)
    Examples::
        >>> m = nn.Linear(20, 30)
        >>> input = autograd.Variable(torch.randn(128, 20))
        >>> output = m(input)
        >>> print(output.size())
    """

    def __init__(self, in_features, out_features, bias=True):
        super(Linear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        # Weight shape: (out_features, in_features)
        self.weight = Parameter(torch.Tensor(out_features, in_features))
        if bias:
            # Bias shape: (out_features,)
            self.bias = Parameter(torch.Tensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        # Initialize weights and bias uniformly
        stdv = 1. / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)

    def forward(self, input):
        # Linear transformation with or without bias
        if self.bias is None:
            return self._backend.Linear()(input, self.weight)
        else:
            return self._backend.Linear()(input, self.weight, self.bias)

    def __repr__(self):
        # String representation for printing
        return self.__class__.__name__ + ' (' \
            + str(self.in_features) + ' -> ' \
            + str(self.out_features) + ')'


#### Custom implementation of a linear layer

In [ ]:
# Custom implementation of a linear layer
class MyLinear(nn.Module):
    
    def __init__(self, in_features, out_features, bias=True):
        super(MyLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        # Weight shape: (in_features, out_features)
        self.weight = Parameter(torch.Tensor(in_features, out_features))
        if bias:
            # Bias shape: (out_features,)
            self.bias = Parameter(torch.Tensor(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, input):
        # Matrix multiplication and optional bias addition
        if self.bias is None:
            return torch.mm(input, self.weight) 
        else:
            return torch.mm(input, self.weight) + self.bias

#### Compare outputs of nn.Linear and custom MyLinear

In [ ]:
# Compare outputs of nn.Linear and custom MyLinear
x = torch.from_numpy(np.random.randn(2, 3)).float()
# Create standard and custom linear layers
linear1 = nn.Linear(3,4)
linear2 = MyLinear(3,4)
# Set the weight and bias of linear2 to match linear1's
linear2.weight.data = linear1.weight.data.transpose(1,0)
linear2.bias.data = linear1.bias.data
# Compare outputs for the same input
print(torch.eq(linear1(x), linear2(x)))

tensor([[ 1,  1,  1,  1],
        [ 1,  1,  1,  1]], dtype=torch.uint8)


### Resnet example

* Resnet blocks let the gradient flow through the hidden unit more directly and at the same time increase expressiveness

Res(x) = F(x, {W}) + x

Res(x) = F(x, {W1}) + W2 x



##### Residual linear layer example for ResNet-like skip connections

In [ ]:
# Residual linear layer example for ResNet-like skip connections
class ResLinear(nn.Module):

    def __init__(self, in_features, out_features, activation=nn.ReLU()):
        super(ResLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.activation = activation
        self.linear = nn.Linear(in_features, out_features)
        # If input and output dimensions differ, add a projection layer
        if in_features != out_features:
            self.project_linear = nn.Linear(in_features, out_features)
        
    def forward(self, x):
        # Apply activation to linear output
        inner = self.activation(self.linear(x))
        # Use projection for skip connection if needed
        if self.in_features != self.out_features:
            skip = self.project_linear(x)
        else:    
            skip = x
        # Add skip connection
        return inner + skip

#### Test standard and residual linear layers

In [ ]:
# Test standard and residual linear layers
x = torch.from_numpy(np.random.randn(2, 3)).float()
res1 = nn.Linear(3,3) # Standard linear layer
res2 = ResLinear(3,5) # Residual linear layer with projection
print(res1(x).size()) # Output shape for standard layer
print(res2(x).size()) # Output shape for residual layer

torch.Size([2, 3])
torch.Size([2, 5])


### Putting things altogether, Sequential, Parameter updates

In [ ]:
# Example model using Sequential and custom layers
class MyModel(nn.Module):
    
    def __init__(self, Linear=ResLinear):
        super(MyModel, self).__init__()
        # Build a sequence of layers for prediction
        self.predict_ = nn.Sequential(
            Linear(784, 328),
            nn.ReLU(),
            Linear(328, 328),
            nn.ReLU(),
            Linear(328, 10),
        )
        self.criterion = nn.CrossEntropyLoss()
    
    def predict_proba(self, x):
        # Softmax for probability output
        return F.softmax(x)
    
    def predict(self, x):
        # Argmax for class prediction
        return torch.max(self.predict_proba(x))[1]
    
    def loss(self, x, target):
        # Compute loss for training
        proba = self.predict_(x)
        return self.criterion(proba, target)
# CrossEntropyLoss expects pre-softmax output
# NLLLoss expects log-softmax output

#### Caveate:    ``CrossEntropyLoss``    versus    ``NLLLoss``


* ``CrossEntropyLoss`` takes in *pre-softmax* as input

* ``NLLLoss`` takes in *log-softmax* as input


In [ ]:
# Compare CrossEntropyLoss and NLLLoss
y = torch.Tensor(1,10).normal_() # Random output
t = torch.from_numpy(np.random.choice(10, size=1)) # Random target
loss1 = nn.CrossEntropyLoss()
loss2 = nn.NLLLoss()
print(loss1(y, t)) # CrossEntropyLoss on raw output
print(loss2(nn.LogSoftmax(dim=1)(y), t)) # NLLLoss on log-softmax output

tensor(2.6893)
tensor(2.6893)


In [ ]:
# Test model loss computation
x = torch.from_numpy(np.random.randn(64, 784)).float() # Random input
t = torch.from_numpy(np.random.choice(10, size=64)) # Random target
model = MyModel()
print(model.loss(x, t)) # Print loss value

tensor(2.3574)


### Updating Parameters (Manually)

In [ ]:
# Manual parameter update example
x = torch.from_numpy(np.random.randn(64, 784)).float() # Random input
t = torch.from_numpy(np.random.choice(10, size=64)) # Random target
model = MyModel()
lr = 0.1 # Learning rate
for i in range(10):
    loss = model.loss(x, t)
    loss.backward()
    for param in model.parameters():
        # Update parameters manually using gradient descent
        param.data.sub_(param.grad.data*lr)
        param.grad.data.zero_()
    print(param.grad) # Print gradients after update

tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])
tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])


### Updating Parameters (``torch.optim``)

In [ ]:
# Parameter update using torch.optim
x = torch.from_numpy(np.random.randn(64, 784)).float() # Random input
t = torch.from_numpy(np.random.choice(10, size=64)) # Random target
model = MyModel()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
for i in range(10):
    optimizer.zero_grad()
    loss = model.loss(x, t)
    loss.backward()
    optimizer.step()
    print(loss) # Print loss after optimizer step
# Momentum helps accelerate updates in the relevant direction

tensor(2.4130)
tensor(1.6219)
tensor(0.6712)
tensor(0.1796)
tensor(1.00000e-02 *
       4.2157)
tensor(1.00000e-02 *
       1.0053)
tensor(1.00000e-03 *
       2.7863)
tensor(1.00000e-04 *
       9.3589)
tensor(1.00000e-04 *
       3.7458)
tensor(1.00000e-04 *
       1.7405)
